In [2]:
!pip install seaborn dotenv


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: C:\Users\natam\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [ ]:
import os
import s3fs
import pandas as pd
import numpy as np
import xarray as xr
import rioxarray
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

# ─── 1. CONFIGURACIÓN Y CREDENCIALES ─────────────────────────────────────────
load_dotenv()
ACCESS_KEY = os.getenv('WASABI_ACCESS_KEY')
SECRET_KEY = os.getenv('WASABI_SECRET_KEY')
REGION     = os.getenv('WASABI_REGION')
BUCKET     = os.getenv('WASABI_BUCKET')

ENDPOINT_URL = f"https://s3.{REGION}.wasabisys.com"  # Para S3FS (Python)
ENDPOINT_GDAL = f"s3.{REGION}.wasabisys.com"         # Para GDAL (C++) sin el https://

# ¡ESTO ES LO QUE FALTABA! Configurar el motor interno (GDAL/rioxarray)
os.environ['AWS_ACCESS_KEY_ID'] = ACCESS_KEY
os.environ['AWS_SECRET_ACCESS_KEY'] = SECRET_KEY
os.environ['AWS_S3_ENDPOINT'] = ENDPOINT_GDAL
os.environ['AWS_VIRTUAL_HOSTING'] = 'FALSE'
os.environ['AWS_HTTPS'] = 'YES'

# ─── 2. CONECTAR A WASABI Y LISTAR ───────────────────────────────────────────
fs = s3fs.S3FileSystem(
    key=ACCESS_KEY,
    secret=SECRET_KEY,
    client_kwargs={'endpoint_url': ENDPOINT_URL}
)

RUTA_MODIS = f"{BUCKET}/MODIS/*/*.tif" # Asegúrate de que esta ruta coincida con tu Wasabi

print("Buscando archivos TIF en Wasabi...")
archivos_tif = fs.glob(RUTA_MODIS)
print(f"Encontrados {len(archivos_tif)} archivos.")

# ─── 3. CARGA PEREZOSA (LAZY LOADING) ────────────────────────────────────────
def cargar_tif(ruta):
    # Extraer la fecha del nombre (Ajustado para formato S2 o MODIS)
    nombre = ruta.split('/')[-1]
    fecha_str = nombre.split('_')[-1].replace('.tif', '')
    fecha = pd.to_datetime(fecha_str)
    
    da = rioxarray.open_rasterio(f"s3://{ruta}", chunks='auto')
    da = da.squeeze('band', drop=True)
    da = da.expand_dims(time=[fecha])
    da.name = "AOD_550nm"
    return da

print("Construyendo Data Cube (Por favor espera unos segundos)...")
datasets = [cargar_tif(f) for f in archivos_tif]
ds_modis = xr.concat(datasets, dim='time').sortby('time')

print("✅ Data Cube de MODIS listo para el EDA:")
print(ds_modis)

Buscando archivos TIF en Wasabi...
Encontrados 1827 archivos.
Construyendo Data Cube (Por favor espera unos segundos)...


In [ ]:
plt.figure(figsize=(10, 8))

# Calculamos la mediana ignorando los NaNs (nubes)
# .compute() le dice a Dask que ahora sí descargue y calcule esto
mapa_media = ds_modis.median(dim='time', skipna=True).compute()

# Graficar
ax = plt.axes()
im = mapa_media.plot.imshow(
    ax=ax,
    cmap='YlOrRd',  # Colores de contaminación (Amarillo a Rojo oscuro)
    add_colorbar=True,
    cbar_kwargs={'label': 'Espesor Óptico de Aerosoles (AOD 550nm)'}
)

ax.set_title("Mapa de Cobertura Espacial Efectiva (AOD MODIS 2020-2024)", fontsize=14)
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))

# Calculamos la mediana ignorando los NaNs (nubes)
# .compute() le dice a Dask que ahora sí descargue y calcule esto
mapa_media = ds_modis.median(dim='time', skipna=True).compute()

# Graficar
ax = plt.axes()
im = mapa_media.plot.imshow(
    ax=ax,
    cmap='YlOrRd',  # Colores de contaminación (Amarillo a Rojo oscuro)
    add_colorbar=True,
    cbar_kwargs={'label': 'Espesor Óptico de Aerosoles (AOD 550nm)'}
)

ax.set_title("Mapa de Cobertura Espacial Efectiva (AOD MODIS 2020-2024)", fontsize=14)
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")

plt.tight_layout()
plt.show()